## 1. Load Data and Rebuild Temporal Splits

### Goal
Load the existing processed Parquet dataset and recreate our training, validation, and test sets.

### Split strategy
- **Fit:** January–August 2024
- **Validation:** September 2024
- **Test:** October–December 2024

We use dates rather than a random split to better reflect how the model will perform on future flights.

The test set remains untouched during tuning.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the existing processed dataset
data_path = Path("../data/processed/flights_2024.parquet")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {data_path.resolve()}\n"
        "Check the notebook's location and update data_path if needed."
    )

df = pd.read_parquet(data_path)
df["FlightDate"] = pd.to_datetime(df["FlightDate"])

# Recreate chronological splits
fit_df = df[df["FlightDate"].dt.month.between(1, 8)].copy()
val_df = df[df["FlightDate"].dt.month == 9].copy()
test_df = df[df["FlightDate"].dt.month.between(10, 12)].copy()

print(f"Dataset shape: {df.shape}")
print(f"Fit:        {fit_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test:       {test_df.shape}")

print("\nDelay rate (DepDel15):")
print(f"Fit:        {fit_df['DepDel15'].mean():.2%}")
print(f"Validation: {val_df['DepDel15'].mean():.2%}")
print(f"Test:       {test_df['DepDel15'].mean():.2%}")

Dataset shape: (6986091, 18)
Fit:        (4638058, 18)
Validation: (579201, 18)
Test:       (1768832, 18)

Delay rate (DepDel15):
Fit:        22.93%
Validation: 15.08%
Test:       16.53%


## 2. Rebuild Features and Train the Baseline

### Goal
Recreate the 13 pre-departure features, align categorical values between fit and validation data, and train the original XGBoost baseline.

### Why this matters
A fresh notebook has no trained model or predictions. We must recreate them before calculating PR-AUC.

### Leakage check
We predict `DepDel15` using only information available before departure. Actual departure times, delays, arrival information, and delay causes must not enter the features.

We will record training time and process memory. Validation will guide early stopping; the test set will not be used.

In [2]:
import os
import time
import psutil
import xgboost as xgb

from sklearn.metrics import average_precision_score

features = [
    "FlightMonth",
    "FlightDay",
    "FlightDayOfWeek",
    "DepHour",
    "DepMinute",
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Distance",
    "ScheduledDuration",
    "DistancePerMinute",
]

target = "DepDel15"
categorical_cols = [
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
]

# Confirm required columns exist
missing = [c for c in features + [target] if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# Build fit and validation matrices
X_fit = fit_df[features].copy()
y_fit = fit_df[target].astype("int8")

X_val = val_df[features].copy()
y_val = val_df[target].astype("int8")

# Normalize the single midnight edge case, if present
for X, split_df in [(X_fit, fit_df), (X_val, val_df)]:
    midnight = split_df["CRSDepTime"].eq(2400)
    X.loc[midnight, "DepHour"] = 0
    X.loc[midnight, "DepMinute"] = 0
    X.loc[midnight, "DepTimeBlock"] = "Night"

# Align categories to fit data; unseen validation categories become missing
for col in categorical_cols:
    X_fit[col] = X_fit[col].astype("category")
    categories = X_fit[col].cat.categories
    X_val[col] = pd.Categorical(X_val[col], categories=categories)

# Confirm the feature list contains no known leakage columns
forbidden = {
    "DepTime", "DepDelay", "ArrTime", "ArrDelay",
    "TaxiOut", "WheelsOff", "WheelsOn",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
}
assert not (set(features) & forbidden), "Potential target leakage!"

# Train the original baseline
process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

train_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

val_scores = model.predict_proba(X_val)[:, 1]
baseline_val_pr_auc = average_precision_score(y_val, val_scores)

print(f"\nBaseline validation PR-AUC: {baseline_val_pr_auc:.4f}")
print(f"Best iteration: {model.best_iteration}")
print(f"Training time: {train_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after:  {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

[0]	validation_0-aucpr:0.23501
[72]	validation_0-aucpr:0.23823

Baseline validation PR-AUC: 0.2442
Best iteration: 22
Training time: 1.33 minutes
Memory before: 948.1 MB
Memory after:  1064.0 MB
Memory change: +115.9 MB


## 3. Tune Model Complexity

### Goal
Find a simpler XGBoost model that performs at least as well on unseen validation data while reducing overfitting.

### Approach
- Train 8 configurations on a fixed 250,000-row sample.
- Test shallower trees, stronger regularization, and different learning rates.
- Use September validation PR-AUC and early stopping to compare configurations.
- Calculate the training–validation PR-AUC gap as an overfitting indicator.

A smaller gap is generally encouraging, but validation PR-AUC remains the main selection metric. We will retrain the selected configuration on the full fit set before final evaluation.

In [3]:
import time
import psutil
import os
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

# Use the same sample for every configuration
sampled_X_fit = X_fit.sample(n=250_000, random_state=42)
sampled_y_fit = y_fit.loc[sampled_X_fit.index]

# Baseline gap measured on that same sample
baseline_train_scores = model.predict_proba(sampled_X_fit)[:, 1]
baseline_train_pr_auc = average_precision_score(
    sampled_y_fit, baseline_train_scores
)
baseline_gap = baseline_train_pr_auc - baseline_val_pr_auc

configs = [
    {"max_depth": 3, "min_child_weight": 20, "learning_rate": 0.03, "reg_lambda": 10},
    {"max_depth": 3, "min_child_weight": 50, "learning_rate": 0.03, "reg_lambda": 20},
    {"max_depth": 4, "min_child_weight": 20, "learning_rate": 0.03, "reg_lambda": 10},
    {"max_depth": 4, "min_child_weight": 50, "learning_rate": 0.03, "reg_lambda": 20},
    {"max_depth": 3, "min_child_weight": 20, "learning_rate": 0.02, "reg_lambda": 20},
    {"max_depth": 4, "min_child_weight": 20, "learning_rate": 0.02, "reg_lambda": 30},
    {"max_depth": 3, "min_child_weight": 50, "learning_rate": 0.05, "reg_lambda": 30},
    {"max_depth": 4, "min_child_weight": 50, "learning_rate": 0.02, "reg_lambda": 50},
]

results = []
candidate_models = []

for i, params in enumerate(configs, start=1):
    candidate = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
        **params,
    )

    candidate.fit(
        sampled_X_fit,
        sampled_y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    train_scores = candidate.predict_proba(sampled_X_fit)[:, 1]
    val_scores_candidate = candidate.predict_proba(X_val)[:, 1]

    train_ap = average_precision_score(sampled_y_fit, train_scores)
    val_ap = average_precision_score(y_val, val_scores_candidate)

    results.append({
        **params,
        "best_iteration": candidate.best_iteration,
        "train_pr_auc": train_ap,
        "val_pr_auc": val_ap,
        "train_val_gap": train_ap - val_ap,
    })
    candidate_models.append(candidate)

    print(
        f"Run {i}/8 | validation PR-AUC: {val_ap:.4f} | "
        f"best iteration: {candidate.best_iteration}"
    )

tuning_results = pd.DataFrame(results).sort_values(
    "val_pr_auc", ascending=False
).reset_index(drop=True)

elapsed_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

print(f"\nBaseline validation PR-AUC: {baseline_val_pr_auc:.4f}")
print(f"Baseline sampled training PR-AUC: {baseline_train_pr_auc:.4f}")
print(f"Baseline sampled train-validation gap: {baseline_gap:.4f}")

display(tuning_results.round(4))

print(f"\nSweep time: {elapsed_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after: {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

Run 1/8 | validation PR-AUC: 0.2367 | best iteration: 146
Run 2/8 | validation PR-AUC: 0.2358 | best iteration: 60
Run 3/8 | validation PR-AUC: 0.2354 | best iteration: 14
Run 4/8 | validation PR-AUC: 0.2369 | best iteration: 115
Run 5/8 | validation PR-AUC: 0.2369 | best iteration: 184
Run 6/8 | validation PR-AUC: 0.2368 | best iteration: 54
Run 7/8 | validation PR-AUC: 0.2369 | best iteration: 67
Run 8/8 | validation PR-AUC: 0.2360 | best iteration: 27

Baseline validation PR-AUC: 0.2442
Baseline sampled training PR-AUC: 0.4331
Baseline sampled train-validation gap: 0.1889


,max_depth,min_child_weight,learning_rate,reg_lambda,best_iteration,train_pr_auc,val_pr_auc,train_val_gap
0,3,50,0.05,30,67,0.4165,0.2369,0.1796
1,3,20,0.02,20,184,0.4189,0.2369,0.1820
2,4,50,0.03,20,115,0.4332,0.2369,0.1963
3,4,20,0.02,30,54,0.4088,0.2368,0.1720
4,3,20,0.03,10,146,0.4245,0.2367,0.1878
5,4,50,0.02,50,27,0.3978,0.2360,0.1618
6,3,50,0.03,20,60,0.4062,0.2358,0.1704
7,4,20,0.03,10,14,0.4004,0.2354,0.1650



Sweep time: 2.43 minutes
Memory before: 889.4 MB
Memory after: 1084.7 MB
Memory change: +195.3 MB


In [4]:
display(tuning_results.round(4))

,max_depth,min_child_weight,learning_rate,reg_lambda,best_iteration,train_pr_auc,val_pr_auc,train_val_gap
0,3,50,0.05,30,67,0.4165,0.2369,0.1796
1,3,20,0.02,20,184,0.4189,0.2369,0.1820
2,4,50,0.03,20,115,0.4332,0.2369,0.1963
3,4,20,0.02,30,54,0.4088,0.2368,0.1720
4,3,20,0.03,10,146,0.4245,0.2367,0.1878
5,4,50,0.02,50,27,0.3978,0.2360,0.1618
6,3,50,0.03,20,60,0.4062,0.2358,0.1704
7,4,20,0.03,10,14,0.4004,0.2354,0.1650


## 4. Refit the Selected Model

### Goal
Train the best candidate on the full fit set and select its classification threshold using validation data.

### Why this configuration?
It achieved the highest validation PR-AUC among the 8 tuning candidates. The alternative with the smallest gap performed worse on validation.

### Evaluation rules
- Fit on January–August.
- Use September for early stopping and threshold selection.
- Do not evaluate October–December until the final evaluation step.

We will compare the new validation score and sampled training–validation gap against the baseline.

In [5]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import time
import psutil
import os
import xgboost as xgb

process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=50,
    reg_lambda=30,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

final_model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

train_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

# Validation evaluation only
val_scores_final = final_model.predict_proba(X_val)[:, 1]
val_pr_auc_final = average_precision_score(y_val, val_scores_final)

# Compare training and validation using the same fixed fit sample
train_scores_final = final_model.predict_proba(sampled_X_fit)[:, 1]
train_pr_auc_final = average_precision_score(
    sampled_y_fit, train_scores_final
)
gap_final = train_pr_auc_final - val_pr_auc_final

# Select threshold by maximizing F1 on validation
precision, recall, thresholds = precision_recall_curve(
    y_val, val_scores_final
)
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"\nFinal best iteration: {final_model.best_iteration}")
print(f"Final validation PR-AUC: {val_pr_auc_final:.4f}")
print(f"Validation-selected threshold: {best_threshold:.4f}")
print(f"Validation precision at threshold: {precision[best_idx]:.4f}")
print(f"Validation recall at threshold: {recall[best_idx]:.4f}")
print(f"Validation F1 at threshold: {f1_scores[best_idx]:.4f}")

print(f"\nBaseline sampled train PR-AUC: {baseline_train_pr_auc:.4f}")
print(f"Baseline validation PR-AUC:    {baseline_val_pr_auc:.4f}")
print(f"Baseline sampled gap:          {baseline_gap:.4f}")
print(f"Final sampled train PR-AUC:    {train_pr_auc_final:.4f}")
print(f"Final validation PR-AUC:       {val_pr_auc_final:.4f}")
print(f"Final sampled gap:             {gap_final:.4f}")

print(f"\nFull refit time: {train_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after:  {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

[0]	validation_0-aucpr:0.21648
[100]	validation_0-aucpr:0.24046
[131]	validation_0-aucpr:0.23753

Final best iteration: 101
Final validation PR-AUC: 0.2407
Validation-selected threshold: 0.2716
Validation precision at threshold: 0.2287
Validation recall at threshold: 0.5664
Validation F1 at threshold: 0.3258

Baseline sampled train PR-AUC: 0.4331
Baseline validation PR-AUC:    0.2442
Baseline sampled gap:          0.1889
Final sampled train PR-AUC:    0.4082
Final validation PR-AUC:       0.2407
Final sampled gap:             0.1675

Full refit time: 0.86 minutes
Memory before: 1084.9 MB
Memory after:  1286.0 MB
Memory change: +201.1 MB


## 5. Final Test Evaluation

### Goal
Evaluate the selected model on October–December 2024, after completing model selection using validation data.

### Metrics
- PR-AUC and lift over the test-set delay prevalence
- ROC-AUC
- Precision, recall, and F1 using the threshold selected on validation

The test set is used only for this final evaluation, not for tuning or threshold selection.

In [6]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

# Prepare test features using the fit-set category definitions
X_test = test_df[features].copy()
y_test = test_df[target].astype("int8")

midnight_test = test_df["CRSDepTime"].eq(2400)
X_test.loc[midnight_test, "DepHour"] = 0
X_test.loc[midnight_test, "DepMinute"] = 0
X_test.loc[midnight_test, "DepTimeBlock"] = "Night"

for col in categorical_cols:
    X_test[col] = pd.Categorical(
        X_test[col],
        categories=X_fit[col].cat.categories,
    )

# One final test evaluation
test_scores_final = final_model.predict_proba(X_test)[:, 1]
test_pr_auc_final = average_precision_score(y_test, test_scores_final)
test_prevalence = y_test.mean()
test_lift = test_pr_auc_final / test_prevalence
test_roc_auc = roc_auc_score(y_test, test_scores_final)

test_predictions = (test_scores_final >= best_threshold).astype("int8")

print(f"Test prevalence: {test_prevalence:.4f}")
print(f"Test PR-AUC: {test_pr_auc_final:.4f}")
print(f"Test PR-AUC lift: {test_lift:.3f}x")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print(f"Test precision: {precision_score(y_test, test_predictions):.4f}")
print(f"Test recall: {recall_score(y_test, test_predictions):.4f}")
print(f"Test F1: {f1_score(y_test, test_predictions):.4f}")
print(f"Validation-selected threshold: {best_threshold:.4f}")

Test prevalence: 0.1653
Test PR-AUC: 0.2443
Test PR-AUC lift: 1.478x
Test ROC-AUC: 0.6389
Test precision: 0.2359
Test recall: 0.5342
Test F1: 0.3273
Validation-selected threshold: 0.2716


## 6. Monthly Test Breakdown

### Goal
Check whether model performance is consistent across October, November, and December.

For each month, calculate PR-AUC, delay prevalence, and PR-AUC lift over that month's prevalence. This is a diagnostic breakdown, not another tuning step.

In [7]:
monthly_results = []

test_months = test_df["FlightDate"].dt.month.to_numpy()
y_test_array = np.asarray(y_test)

for month in [10, 11, 12]:
    mask = test_months == month

    month_y = y_test_array[mask]
    month_scores = test_scores_final[mask]

    prevalence = month_y.mean()
    pr_auc = average_precision_score(month_y, month_scores)

    monthly_results.append({
        "month": {10: "October", 11: "November", 12: "December"}[month],
        "rows": int(mask.sum()),
        "delay_prevalence": prevalence,
        "pr_auc": pr_auc,
        "lift": pr_auc / prevalence,
    })

monthly_results = pd.DataFrame(monthly_results)
display(monthly_results.round(4))

,month,rows,delay_prevalence,pr_auc,lift
0,October,609295,0.1361,0.2141,1.5736
1,November,572872,0.1441,0.2141,1.4851
2,December,586665,0.2165,0.3057,1.4119


## Baseline Stability Check

### Goal
Measure how much validation PR-AUC changes across three random seeds while keeping the dataset, chronological split, features, and baseline hyperparameters unchanged.

### Why
A tiny score difference may be caused by training randomness rather than a genuine model improvement.

We will use validation PR-AUC only. The test set will not be used for this comparison.

In [8]:
import time
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

seeds = [21, 42, 84]
seed_results = []

for seed in seeds:
    model_seed = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1,
    )

    start = time.perf_counter()

    model_seed.fit(
        X_fit,
        y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_scores_seed = model_seed.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_scores_seed)

    seed_results.append({
        "seed": seed,
        "best_iteration": model_seed.best_iteration,
        "validation_pr_auc": val_ap,
        "training_minutes": (time.perf_counter() - start) / 60,
    })

seed_results = pd.DataFrame(seed_results)
display(seed_results.round(4))

print(f"Mean validation PR-AUC: {seed_results['validation_pr_auc'].mean():.4f}")
print(f"Score range: {seed_results['validation_pr_auc'].min():.4f}"
      f"–{seed_results['validation_pr_auc'].max():.4f}")
print(f"Range width: {seed_results['validation_pr_auc'].max() - seed_results['validation_pr_auc'].min():.4f}")

,seed,best_iteration,validation_pr_auc,training_minutes
0,21,38,0.2448,1.1154
1,42,22,0.2442,0.9593
2,84,5,0.2445,0.7814


Mean validation PR-AUC: 0.2445
Score range: 0.2442–0.2448
Range width: 0.0006


## Primary Model Decision

**Chosen model:** Original XGBoost baseline (`max_depth=8`, `min_child_weight=10`, `learning_rate=0.05`, `reg_lambda=5`).

**Reason:** Baseline validation PR-AUC averaged 0.2445 across three seeds, with a 0.0006 range. The tuned model's 0.0006 test PR-AUC advantage is too small to establish a reliable improvement, and its validation PR-AUC was lower (0.2407 vs. 0.2442). Keep the baseline as primary and retain the tuned model as an experiment.

---

In [9]:
# Weather pilot: August 2024, using the 10 busiest origin airports

pilot_month = df.loc[df["FlightDate"].dt.month == 8].copy()

airport_counts = (
    pilot_month["Origin"]
    .value_counts()
    .head(10)
    .rename_axis("Origin")
    .reset_index(name="flight_count")
)

print(f"August 2024 flight rows: {len(pilot_month):,}")
display(airport_counts)

August 2024 flight rows: 606,942


,Origin,flight_count
0,ATL,29187
1,DFW,27632
2,DEN,27148
3,ORD,25167
4,CLT,18559
5,LAX,17049
6,LAS,15963
7,SEA,15861
8,PHX,15315
9,LGA,13143


In [10]:
# Inspect fields needed to calculate the T-2-hour weather cutoff

time_columns = [
    c for c in df.columns
    if any(term in c.lower() for term in ["date", "time", "origin"])
]

print("Potential date/time columns:")
print(time_columns)

display(pilot_month[
    ["FlightDate", "Origin", "CRSDepTime"]
].head(10))

print("\nCRSDepTime examples and data types:")
print(pilot_month["CRSDepTime"].dtype)
print(pilot_month["CRSDepTime"].value_counts().head(10))

Potential date/time columns:
['FlightDate', 'Origin', 'CRSDepTime', 'CRSElapsedTime', 'DepTimeBlock']


,FlightDate,Origin,CRSDepTime
5799948,2024-08-09,PHL,1051
5799949,2024-08-10,PHL,1051
5799950,2024-08-11,PHL,1051
5799951,2024-08-12,PHL,1051
5799952,2024-08-13,PHL,1051
5799953,2024-08-14,PHL,1051
5799954,2024-08-15,PHL,1051
5799955,2024-08-16,PHL,1051
5799956,2024-08-17,PHL,1051
5799957,2024-08-18,PHL,1051



CRSDepTime examples and data types:
int16
CRSDepTime
600     13597
700     10506
800      5444
900      4089
630      3705
830      3672
1000     3661
715      3409
1100     3213
710      3126
Name: count, dtype: int64


In [11]:
# Check scheduled-time fields and pilot airport coverage

print("Scheduled departure fields:")
print([
    c for c in pilot_month.columns
    if c in ["FlightDate", "CRSDepTime", "CRSDepDate", "Origin"]
])

print("\nPilot airports:")
display(airport_counts)

print("\nSample scheduled dates and times:")
display(
    pilot_month[["FlightDate", "CRSDepTime", "Origin"]]
    .head(10)
)

Scheduled departure fields:
['FlightDate', 'Origin', 'CRSDepTime']

Pilot airports:


,Origin,flight_count
0,ATL,29187
1,DFW,27632
2,DEN,27148
3,ORD,25167
4,CLT,18559
5,LAX,17049
6,LAS,15963
7,SEA,15861
8,PHX,15315
9,LGA,13143



Sample scheduled dates and times:


,FlightDate,CRSDepTime,Origin
5799948,2024-08-09,1051,PHL
5799949,2024-08-10,1051,PHL
5799950,2024-08-11,1051,PHL
5799951,2024-08-12,1051,PHL
5799952,2024-08-13,1051,PHL
5799953,2024-08-14,1051,PHL
5799954,2024-08-15,1051,PHL
5799955,2024-08-16,1051,PHL
5799956,2024-08-17,1051,PHL
5799957,2024-08-18,1051,PHL


In [12]:
# Restrict the pilot to the 10 busiest August origin airports
pilot_airports = airport_counts["Origin"].tolist()

pilot_flights = pilot_month[
    pilot_month["Origin"].isin(pilot_airports)
].copy()

# Check HHMM values before converting to timestamps
hhmm = pilot_flights["CRSDepTime"]

invalid_time = (
    (hhmm < 0)
    | (hhmm > 2400)
    | ((hhmm % 100) >= 60)
    | ((hhmm == 2400) & (hhmm % 100 != 0))
)

print(f"Pilot flight rows: {len(pilot_flights):,}")
print(f"Airports included: {pilot_flights['Origin'].nunique()}")
print(f"Invalid scheduled-time rows: {invalid_time.sum():,}")
print(f"2400 (midnight) rows: {(hhmm == 2400).sum():,}")

display(
    pilot_flights.loc[invalid_time, ["FlightDate", "Origin", "CRSDepTime"]]
    .head(10)
)

Pilot flight rows: 205,024
Airports included: 10
Invalid scheduled-time rows: 0
2400 (midnight) rows: 0


,FlightDate,Origin,CRSDepTime


In [17]:
# Airport time zones for the 10-airport weather pilot

airport_timezones = {
    "ATL": "America/New_York",
    "DFW": "America/Chicago",
    "DEN": "America/Denver",
    "ORD": "America/Chicago",
    "CLT": "America/New_York",
    "LAX": "America/Los_Angeles",
    "LAS": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "PHX": "America/Phoenix",
    "LGA": "America/New_York",
}

assert set(pilot_airports) == set(airport_timezones), (
    "Time-zone mapping does not match pilot airports"
)

display(pd.DataFrame([
    {"Origin": airport, "timezone": timezone}
    for airport, timezone in airport_timezones.items()
]))

,Origin,timezone
0,ATL,America/New_York
1,DFW,America/Chicago
2,DEN,America/Denver
3,ORD,America/Chicago
4,CLT,America/New_York
5,LAX,America/Los_Angeles
6,LAS,America/Los_Angeles
7,SEA,America/Los_Angeles
8,PHX,America/Phoenix
9,LGA,America/New_York


In [21]:
import pandas as pd

# 1. Parse hours and minutes globally
hhmm = pilot_flights["CRSDepTime"].astype(int)
hours = hhmm // 100
minutes = hhmm % 100

# 2. Build a clean, naive local datetime series
pilot_flights["scheduled_departure_local"] = (
    pilot_flights["FlightDate"].dt.normalize()
    + pd.to_timedelta(hours, unit="h")
    + pd.to_timedelta(minutes, unit="m")
)

# 3. Vectorized Magic: Map timezones to rows and group-localize to UTC
# This computes everything safely without touching the main DataFrame until it's ready
pilot_flights["scheduled_departure_utc"] = (
    pilot_flights.groupby(pilot_flights["Origin"].map(airport_timezones), group_keys=False)
    ["scheduled_departure_local"]
    .apply(lambda x: x.dt.tz_localize(x.name, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC"))
)

# 4. Calculate the T-2h weather cutoff
pilot_flights["weather_cutoff_utc"] = (
    pilot_flights["scheduled_departure_utc"] - pd.Timedelta(hours=2)
)

# --- Diagnostics & Display ---
print("Rows:", len(pilot_flights))
print("Missing UTC departure:", pilot_flights["scheduled_departure_utc"].isna().sum())
print("Missing weather cutoff:", pilot_flights["weather_cutoff_utc"].isna().sum())

display(
    pilot_flights[
        [
            "FlightDate",
            "Origin",
            "CRSDepTime",
            "scheduled_departure_local",
            "scheduled_departure_utc",
            "weather_cutoff_utc",
        ]
    ].head(10)
)


Rows: 205024
Missing UTC departure: 0
Missing weather cutoff: 0


,FlightDate,Origin,CRSDepTime,scheduled_departure_local,scheduled_departure_utc,weather_cutoff_utc
5799971,2024-08-06,LAX,1030,2024-08-06 10:30:00,2024-08-06 17:30:00+00:00,2024-08-06 15:30:00+00:00
5799972,2024-08-07,LAX,1030,2024-08-07 10:30:00,2024-08-07 17:30:00+00:00,2024-08-07 15:30:00+00:00
5799973,2024-08-08,LAX,1030,2024-08-08 10:30:00,2024-08-08 17:30:00+00:00,2024-08-08 15:30:00+00:00
5799974,2024-08-09,LAX,1030,2024-08-09 10:30:00,2024-08-09 17:30:00+00:00,2024-08-09 15:30:00+00:00
5799975,2024-08-10,LAX,1030,2024-08-10 10:30:00,2024-08-10 17:30:00+00:00,2024-08-10 15:30:00+00:00
5799976,2024-08-11,LAX,1030,2024-08-11 10:30:00,2024-08-11 17:30:00+00:00,2024-08-11 15:30:00+00:00
5799977,2024-08-12,LAX,1030,2024-08-12 10:30:00,2024-08-12 17:30:00+00:00,2024-08-12 15:30:00+00:00
5799978,2024-08-13,LAX,1030,2024-08-13 10:30:00,2024-08-13 17:30:00+00:00,2024-08-13 15:30:00+00:00
5799979,2024-08-14,LAX,1030,2024-08-14 10:30:00,2024-08-14 17:30:00+00:00,2024-08-14 15:30:00+00:00
5799980,2024-08-15,LAX,1030,2024-08-15 10:30:00,2024-08-15 17:30:00+00:00,2024-08-15 15:30:00+00:00


In [22]:
# Build scheduled departure timestamps and T-2h weather cutoffs in UTC

utc_departures = pd.Series(
    pd.NaT,
    index=pilot_flights.index,
    dtype="datetime64[ns, UTC]",
)

for airport, timezone in airport_timezones.items():
    mask = pilot_flights["Origin"].eq(airport)

    hhmm = pilot_flights.loc[mask, "CRSDepTime"].astype(int)

    local_naive = (
        pilot_flights.loc[mask, "FlightDate"].dt.normalize()
        + pd.to_timedelta(hhmm // 100, unit="h")
        + pd.to_timedelta(hhmm % 100, unit="m")
    )

    local_aware = local_naive.dt.tz_localize(
        timezone,
        ambiguous="NaT",
        nonexistent="NaT",
    )

    utc_departures.loc[mask] = local_aware.dt.tz_convert("UTC")

pilot_flights["scheduled_departure_utc"] = utc_departures
pilot_flights["weather_cutoff_utc"] = (
    pilot_flights["scheduled_departure_utc"] - pd.Timedelta(hours=2)
)

print(f"Pilot rows: {len(pilot_flights):,}")
print(f"Missing UTC departure: {utc_departures.isna().sum():,}")
print(f"Missing weather cutoff: {pilot_flights['weather_cutoff_utc'].isna().sum():,}")

display(
    pilot_flights[
        ["FlightDate", "Origin", "CRSDepTime",
         "scheduled_departure_utc", "weather_cutoff_utc"]
    ].head(10)
)

Pilot rows: 205,024
Missing UTC departure: 0
Missing weather cutoff: 0


,FlightDate,Origin,CRSDepTime,scheduled_departure_utc,weather_cutoff_utc
5799971,2024-08-06,LAX,1030,2024-08-06 17:30:00+00:00,2024-08-06 15:30:00+00:00
5799972,2024-08-07,LAX,1030,2024-08-07 17:30:00+00:00,2024-08-07 15:30:00+00:00
5799973,2024-08-08,LAX,1030,2024-08-08 17:30:00+00:00,2024-08-08 15:30:00+00:00
5799974,2024-08-09,LAX,1030,2024-08-09 17:30:00+00:00,2024-08-09 15:30:00+00:00
5799975,2024-08-10,LAX,1030,2024-08-10 17:30:00+00:00,2024-08-10 15:30:00+00:00
5799976,2024-08-11,LAX,1030,2024-08-11 17:30:00+00:00,2024-08-11 15:30:00+00:00
5799977,2024-08-12,LAX,1030,2024-08-12 17:30:00+00:00,2024-08-12 15:30:00+00:00
5799978,2024-08-13,LAX,1030,2024-08-13 17:30:00+00:00,2024-08-13 15:30:00+00:00
5799979,2024-08-14,LAX,1030,2024-08-14 17:30:00+00:00,2024-08-14 15:30:00+00:00
5799980,2024-08-15,LAX,1030,2024-08-15 17:30:00+00:00,2024-08-15 15:30:00+00:00


---

In [23]:

# Slot 3A: Map the 10 pilot airports to NOAA ISD stations
from pathlib import Path
import pandas as pd
import numpy as np

weather_dir = Path("../data/external/noaa_isd")
weather_dir.mkdir(parents=True, exist_ok=True)

# Airport coordinates (latitude, longitude)
airport_coords = {
    "ATL": (33.6407, -84.4277),
    "DFW": (32.8998, -97.0403),
    "DEN": (39.8561, -104.6737),
    "ORD": (41.9742, -87.9073),
    "CLT": (35.2140, -80.9431),
    "LAX": (33.9416, -118.4085),
    "LAS": (36.0840, -115.1537),
    "SEA": (47.4502, -122.3088),
    "PHX": (33.4342, -112.0116),
    "LGA": (40.7769, -73.8740),
}

assert set(pilot_airports) == set(airport_coords), (
    f"Unexpected pilot airports: {pilot_airports}"
)

inventory_url = (
    "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
)
inventory_path = weather_dir / "isd-history.csv"

if not inventory_path.exists():
    inventory_path.write_bytes(
        __import__("requests").get(
            inventory_url, timeout=60
        ).content
    )

stations = pd.read_csv(inventory_path, dtype={"USAF": str, "WBAN": str})
stations.columns = stations.columns.str.strip()

for col in ["LAT", "LON", "BEGIN", "END"]:
    stations[col] = pd.to_numeric(stations[col], errors="coerce")

stations["USAF"] = stations["USAF"].str.strip().str.zfill(6)
stations["WBAN"] = stations["WBAN"].str.strip().str.zfill(5)
stations = stations[
    stations["LAT"].notna()
    & stations["LON"].notna()
    & stations["USAF"].str.fullmatch(r"\d{6}")
    & stations["WBAN"].str.fullmatch(r"\d{5}")
].copy()

# Retain stations whose inventory period includes August 2024.
stations = stations[
    (stations["BEGIN"] <= 20240801)
    & (stations["END"] >= 20240831)
].copy()

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2)**2
    return 2 * radius * np.arcsin(np.sqrt(a))

mapping = []

for airport in pilot_airports:
    lat, lon = airport_coords[airport]
    candidates = stations.copy()
    candidates["distance_km"] = haversine_km(
        lat, lon, candidates["LAT"], candidates["LON"]
    )
    best = candidates.sort_values("distance_km").iloc[0]

    mapping.append({
        "Origin": airport,
        "station_id": best["USAF"] + best["WBAN"],
        "station_name": best.get("STATION NAME", ""),
        "station_lat": best["LAT"],
        "station_lon": best["LON"],
        "distance_km": best["distance_km"],
    })

station_map = pd.DataFrame(mapping)
display(station_map.round({"distance_km": 2}))


,Origin,station_id,station_name,station_lat,station_lon,distance_km
0,ATL,72219013874,HARTSFIELD-JACKSON ATLANTA INTL AP,33.630,-84.442,1.78
1,DFW,72259003927,DALLAS/FT WORTH INTERNATIONAL AP,32.897,-97.022,1.74
2,DEN,72565003017,DENVER INTERNATIONAL AIRPORT,39.847,-104.656,1.82
3,ORD,72530094846,CHICAGO O'HARE INTERNATIONAL AIRPORT,41.960,-87.932,2.58
4,CLT,72314013881,CHARLOTTE/DOUGLAS INTERNATIONAL AP,35.223,-80.954,1.41
5,LAX,72295023174,LOS ANGELES INTERNATIONAL AIRPORT,33.938,-118.387,2.02
6,LAS,72386023169,MCCARRAN INTERNATIONAL AIRPORT,36.072,-115.163,1.57
7,SEA,72793024233,SEATTLE-TACOMA INTERNATIONAL AIRPORT,47.445,-122.314,0.70
8,PHX,72278023183,PHOENIX SKY HARBOR INTL AIRPORT,33.428,-112.004,0.99
9,LGA,72503014732,LA GUARDIA AIRPORT,40.779,-73.880,0.56


In [26]:
# CELL 2 — Updated NOAA public-cloud URL

import io
import requests
import pandas as pd

weather_frames = []
download_log = []

for _, station in station_map.iterrows():
    station_id = str(station["station_id"])
    url = f"https://noaa-global-hourly-pds.s3.amazonaws.com/2024/{station_id}.csv"

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        if response.text.lstrip().lower().startswith(
            ("<!doctype html", "<html")
        ):
            raise ValueError("Received HTML instead of CSV")

        df = pd.read_csv(io.StringIO(response.text), dtype=str)

        if "DATE" not in df.columns:
            raise ValueError(f"DATE missing; columns: {list(df.columns)[:5]}")

        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
        df = df[
            (df["DATE"] >= "2024-08-01") &
            (df["DATE"] < "2024-09-01")
        ].copy()

        df["station_id"] = station_id

        if not df.empty:
            weather_frames.append(df)

        download_log.append({
            "station_id": station_id,
            "status": "OK" if not df.empty else "NO AUGUST ROWS",
            "august_rows": len(df),
            "error": "",
        })

    except Exception as e:
        download_log.append({
            "station_id": station_id,
            "status": "FAILED",
            "august_rows": 0,
            "error": str(e)[:250],
        })

weather_downloads = pd.DataFrame(download_log)
display(weather_downloads)

if not weather_frames:
    raise RuntimeError("No NOAA observations loaded. Inspect the error column.")

weather = pd.concat(weather_frames, ignore_index=True)
print(
    f"Loaded {len(weather):,} August observations across "
    f"{weather['station_id'].nunique()} stations."
)

,station_id,status,august_rows,error
0,72219013874,OK,1077,
1,72259003927,OK,1031,
2,72565003017,OK,1156,
3,72530094846,OK,1098,
4,72314013881,OK,1171,
5,72295023174,OK,1128,
6,72386023169,OK,1052,
7,72793024233,OK,1145,
8,72278023183,OK,1085,
9,72503014732,OK,1160,


Loaded 11,103 August observations across 10 stations.


In [28]:

# CELL 3 — Parse NOAA fields, join weather to flights, and measure coverage

import numpy as np
import pandas as pd

weather_fields = ["TMP", "DEW", "SLP", "WND", "VIS", "CIG"]

def parse_noaa_value(series):
    extracted = series.astype("string").str.extract(
        r"([+-]?\d+)", expand=False
    )
    return pd.to_numeric(extracted, errors="coerce")


# 1. Prepare weather observations
available_fields = [c for c in weather_fields if c in weather.columns]

weather_clean = weather[
    ["station_id", "DATE"] + available_fields
].copy()

weather_clean = weather_clean.rename(
    columns={"DATE": "weather_time_utc"}
)

weather_clean["weather_time_utc"] = pd.to_datetime(
    weather_clean["weather_time_utc"], utc=True, errors="coerce"
).astype("datetime64[ns, UTC]")

for col in available_fields:
    weather_clean[col + "_raw"] = weather_clean[col]
    weather_clean[col] = parse_noaa_value(weather_clean[col])

weather_clean["station_id"] = weather_clean["station_id"].astype("string")


# 2. Prepare every pilot flight
pilot = pilot_flights.copy().reset_index(drop=True)
pilot["_flight_id"] = np.arange(len(pilot))

pilot["weather_cutoff_utc"] = pd.to_datetime(
    pilot["weather_cutoff_utc"], utc=True, errors="coerce"
).astype("datetime64[ns, UTC]")

pilot = pilot.merge(
    station_map[["Origin", "station_id"]],
    on="Origin",
    how="left",
)

pilot["station_id"] = pilot["station_id"].astype("string")


# 3. Backward-looking as-of join
left = pilot.dropna(
    subset=["weather_cutoff_utc", "station_id"]
).copy()

right = weather_clean.dropna(
    subset=["weather_time_utc", "station_id"]
).copy()

# Both merge keys must have exactly the same datetime dtype.
left["weather_cutoff_utc"] = left["weather_cutoff_utc"].astype(
    "datetime64[ns, UTC]"
)
right["weather_time_utc"] = right["weather_time_utc"].astype(
    "datetime64[ns, UTC]"
)

left = left.sort_values("weather_cutoff_utc")
right = right.sort_values("weather_time_utc")

joined = pd.merge_asof(
    left,
    right,
    left_on="weather_cutoff_utc",
    right_on="weather_time_utc",
    by="station_id",
    direction="backward",
    tolerance=pd.Timedelta("2h"),
)

joined["weather_age_minutes"] = (
    joined["weather_cutoff_utc"] - joined["weather_time_utc"]
).dt.total_seconds() / 60


# 4. Restore all flights, even those without a weather match
weather_output_cols = [
    c for c in weather_clean.columns
    if c not in ["station_id", "weather_time_utc"]
]

match_cols = [
    "_flight_id",
    "weather_time_utc",
    "weather_age_minutes",
] + weather_output_cols

pilot_weather = pilot.merge(
    joined[match_cols].drop_duplicates("_flight_id"),
    on="_flight_id",
    how="left",
)

matched = pilot_weather["weather_time_utc"].notna()


# 5. Overall join quality
summary = pd.DataFrame([{
    "pilot_flights": len(pilot_weather),
    "flights_with_weather": int(matched.sum()),
    "flights_without_weather": int((~matched).sum()),
    "join_success_pct": round(100 * matched.mean(), 2)
        if len(pilot_weather) else 0,
    "future_timestamp_violations": int((
        pilot_weather["weather_time_utc"]
        > pilot_weather["weather_cutoff_utc"]
    ).fillna(False).sum()),
    "older_than_120min_violations": int((
        pilot_weather["weather_age_minutes"] > 120
    ).fillna(False).sum()),
}])


# 6. Per-field missingness across ALL pilot flights
field_missingness = pd.DataFrame({
    "field": available_fields,
    "missing_pct": [
        round(100 * pilot_weather[c].isna().mean(), 2)
        for c in available_fields
    ],
})


# 7. Station-level coverage
station_coverage = station_map[
    ["Origin", "station_id", "station_name", "distance_km"]
].copy()

station_coverage["station_id"] = (
    station_coverage["station_id"].astype("string")
)

row_counts = (
    weather.groupby("station_id")
    .size()
    .rename("august_rows")
    .reset_index()
)

row_counts["station_id"] = row_counts["station_id"].astype("string")

station_coverage = station_coverage.merge(
    row_counts,
    on="station_id",
    how="left",
)

station_coverage["august_rows"] = (
    station_coverage["august_rows"].fillna(0).astype(int)
)

display(summary)
display(station_coverage)
display(field_missingness)

print(f"Pilot flights: {len(pilot_weather):,}")
print(f"Flights matched to weather: {int(matched.sum()):,}")


,pilot_flights,flights_with_weather,flights_without_weather,join_success_pct,future_timestamp_violations,older_than_120min_violations
0,205024,204858,166,99.92,0,0


,Origin,station_id,station_name,distance_km,august_rows
0,ATL,72219013874,HARTSFIELD-JACKSON ATLANTA INTL AP,1.779953,1077
1,DFW,72259003927,DALLAS/FT WORTH INTERNATIONAL AP,1.736682,1031
2,DEN,72565003017,DENVER INTERNATIONAL AIRPORT,1.818488,1156
3,ORD,72530094846,CHICAGO O'HARE INTERNATIONAL AIRPORT,2.581350,1098
4,CLT,72314013881,CHARLOTTE/DOUGLAS INTERNATIONAL AP,1.407819,1171
5,LAX,72295023174,LOS ANGELES INTERNATIONAL AIRPORT,2.023369,1128
6,LAS,72386023169,MCCARRAN INTERNATIONAL AIRPORT,1.574484,1052
7,SEA,72793024233,SEATTLE-TACOMA INTERNATIONAL AIRPORT,0.698020,1145
8,PHX,72278023183,PHOENIX SKY HARBOR INTL AIRPORT,0.986245,1085
9,LGA,72503014732,LA GUARDIA AIRPORT,0.556566,1160


,field,missing_pct
0,TMP,0.08
1,DEW,0.08
2,SLP,0.08
3,WND,0.08
4,VIS,0.08
5,CIG,0.08


Pilot flights: 205,024
Flights matched to weather: 204,858


In [29]:

# CELL 4 — Validate matched weather values and raw encodings

matched_rows = pilot_weather.loc[
    pilot_weather["weather_time_utc"].notna()
].copy()

print(f"Matched flights: {len(matched_rows):,}")

field_check = pd.DataFrame({
    "field": available_fields,
    "missing_pct_among_matched": [
        round(100 * matched_rows[c].isna().mean(), 3)
        for c in available_fields
    ],
    "unique_values": [
        matched_rows[c].nunique(dropna=True)
        for c in available_fields
    ],
})

display(field_check)

for field in available_fields:
    raw_field = field + "_raw"
    if raw_field in matched_rows.columns:
        print(f"\n{field} raw samples:")
        display(
            matched_rows[raw_field]
            .dropna()
            .astype(str)
            .value_counts()
            .head(8)
            .rename_axis("raw_value")
            .reset_index(name="count")
        )


Matched flights: 204,858


,field,missing_pct_among_matched,unique_values
0,TMP,0.0,90
1,DEW,0.0,84
2,SLP,0.0,257
3,WND,0.0,37
4,VIS,0.0,36
5,CIG,0.0,81



TMP raw samples:


,raw_value,count
0,"+0267,5",5090
1,"+0233,5",4741
2,"+0239,5",4739
3,"+0278,5",4730
4,"+0250,5",4663
5,"+0228,5",4660
6,"+0317,5",4263
7,"+0272,5",4210



DEW raw samples:


...

,raw_value,count
0,"999,9,C,0000,5",11556
1,"999,9,C,0000,1",4762
2,"999,9,V,0015,5",3786
3,"999,9,V,0021,5",2634
4,"999,9,V,0026,5",2192
5,"999,9,N,0015,1",1746
6,"999,9,V,0031,5",1346
7,"260,5,N,0031,5",1084



VIS raw samples:


,raw_value,count
0,"016093,5,N,5",136363
1,"016000,1,9,9",50886
2,"014484,5,N,5",3030
3,"012875,5,N,5",2342
4,"011265,5,N,5",1860
5,"014000,1,9,9",1386
6,"009656,5,N,5",1130
7,"008047,5,N,5",1080



CIG raw samples:


,raw_value,count
0,"22000,5,9,N",96818
1,"22000,1,9,N",28495
2,"99999,9,9,N",26414
3,"07620,5,M,N",10808
4,"06706,5,M,N",2783
5,"04572,5,M,N",2525
6,"06096,5,M,N",2314
7,"04267,5,M,N",2194


In [30]:

# CELL 5 — Decode NOAA fields correctly
# Uses the raw NOAA strings preserved in pilot_weather.

import numpy as np
import pandas as pd

def noaa_primary_value(series):
    """Parse the value before the first comma; honor NOAA missing codes."""
    raw = series.astype("string").str.strip()
    first = raw.str.split(",", regex=False).str[0]

    value = pd.to_numeric(
        first.str.replace("+", "", regex=False),
        errors="coerce",
    )

    # NOAA commonly uses these sentinel values for missing data.
    value = value.mask(first.str.replace("+", "", regex=False).isin(
        ["9999", "99999", "999999"]
    ))
    return value


# Temperature and dew point: tenths of °C
for field in ["TMP", "DEW"]:
    raw_col = field + "_raw"
    if raw_col in pilot_weather.columns:
        pilot_weather[field + "_c"] = noaa_primary_value(
            pilot_weather[raw_col]
        ) / 10

# Sea-level pressure: tenths of hPa; 99999 means missing.
if "SLP_raw" in pilot_weather.columns:
    pilot_weather["SLP_hpa"] = noaa_primary_value(
        pilot_weather["SLP_raw"]
    ) / 10

# Visibility: first value is in meters; 999999 means missing.
if "VIS_raw" in pilot_weather.columns:
    pilot_weather["VIS_m"] = noaa_primary_value(
        pilot_weather["VIS_raw"]
    ).mask(
        noaa_primary_value(pilot_weather["VIS_raw"]) >= 999999
    )

# Ceiling height: first value is in meters; 99999 means missing.
if "CIG_raw" in pilot_weather.columns:
    pilot_weather["CIG_m"] = noaa_primary_value(
        pilot_weather["CIG_raw"]
    ).mask(
        noaa_primary_value(pilot_weather["CIG_raw"]) >= 99999
    )

# Wind: encoded as direction, quality, type, speed, speed quality.
# Preserve the components separately rather than parsing the whole string.
if "WND_raw" in pilot_weather.columns:
    wind_parts = pilot_weather["WND_raw"].astype("string").str.split(
        ",", expand=True
    )

    if wind_parts.shape[1] >= 5:
        direction = pd.to_numeric(wind_parts[0], errors="coerce")
        speed = pd.to_numeric(wind_parts[3], errors="coerce")

        pilot_weather["wind_direction_deg"] = direction.mask(
            direction >= 999
        )
        pilot_weather["wind_speed_ms"] = speed.mask(
            speed >= 9999
        ) / 10

# Report decoded coverage and plausible ranges.
decoded_cols = [
    c for c in [
        "TMP_c", "DEW_c", "SLP_hpa", "VIS_m", "CIG_m",
        "wind_direction_deg", "wind_speed_ms"
    ] if c in pilot_weather.columns
]

display(pd.DataFrame({
    "field": decoded_cols,
    "missing_pct": [
        round(100 * pilot_weather[c].isna().mean(), 2)
        for c in decoded_cols
    ],
    "min": [pilot_weather[c].min() for c in decoded_cols],
    "median": [pilot_weather[c].median() for c in decoded_cols],
    "max": [pilot_weather[c].max() for c in decoded_cols],
}))


,field,missing_pct,min,median,max
0,TMP_c,0.10,10.0,27.2,46.7
1,DEW_c,0.10,-9.4,16.1,25.6
2,SLP_hpa,4.21,1000.7,1014.9,1028.1
3,VIS_m,0.08,200.0,16093.0,16093.0
4,CIG_m,12.96,30.0,22000.0,22000.0
5,wind_direction_deg,14.85,10.0,200.0,360.0
6,wind_speed_ms,0.14,0.0,3.6,16.0


In [33]:
# CELL 6 — Decode NOAA special values and inspect quality flags

import pandas as pd
import numpy as np

def split_raw(field):
    return pilot_weather[field + "_raw"].astype("string").str.split(
        ",", expand=True
    )

def numeric_part(series):
    return pd.to_numeric(series, errors="coerce")


# 1. Temperature and dew point: tenths of °C
for field in ["TMP", "DEW"]:
    raw_col = field + "_raw"
    if raw_col not in pilot_weather.columns:
        continue

    parts = split_raw(field)
    value = numeric_part(
        parts[0].str.replace("+", "", regex=False)
    )

    pilot_weather[field + "_c"] = value.mask(value.abs() >= 9999) / 10
    pilot_weather[field + "_qc"] = (
        parts[1] if parts.shape[1] > 1 else pd.NA
    )


# 2. Sea-level pressure: tenths of hPa
if "SLP_raw" in pilot_weather.columns:
    parts = split_raw("SLP")
    value = numeric_part(parts[0])

    pilot_weather["SLP_hpa"] = value.mask(value >= 99999) / 10
    pilot_weather["SLP_qc"] = (
        parts[1] if parts.shape[1] > 1 else pd.NA
    )


# 3. Visibility: meters plus reporting/quality flags
if "VIS_raw" in pilot_weather.columns:
    parts = split_raw("VIS")
    value = numeric_part(parts[0])

    pilot_weather["VIS_m"] = value.mask(value >= 999999)
    pilot_weather["VIS_qc"] = parts[1]
    pilot_weather["VIS_flag"] = parts[2]

    # Missing flags become False safely; 16093 m is about 10 miles.
    pilot_weather["VIS_at_least_10mi"] = (
        pilot_weather["VIS_flag"].eq("N").fillna(False)
        | pilot_weather["VIS_m"].ge(16093).fillna(False)
    ).astype("int8")


# 4. Ceiling: 22000 is a special unlimited-ceiling code
if "CIG_raw" in pilot_weather.columns:
    parts = split_raw("CIG")
    value = numeric_part(parts[0])

    pilot_weather["CIG_unlimited"] = (
        value.eq(22000).fillna(False).astype("int8")
    )
    pilot_weather["CIG_m"] = value.mask(
        (value >= 99999) | value.eq(22000)
    )

    pilot_weather["CIG_qc"] = parts[1]
    pilot_weather["CIG_determination"] = parts[2]
    pilot_weather["CIG_cavok"] = parts[3]


# 5. Wind: direction, speed, and separate quality codes
if "WND_raw" in pilot_weather.columns:
    parts = split_raw("WND")

    direction = numeric_part(parts[0])
    speed = numeric_part(parts[3])

    pilot_weather["wind_direction_deg"] = direction.mask(
        direction >= 999
    )
    pilot_weather["wind_direction_qc"] = parts[1]
    pilot_weather["wind_type"] = parts[2]

    pilot_weather["wind_speed_ms"] = speed.mask(
        speed >= 9999
    ) / 10
    pilot_weather["wind_speed_qc"] = parts[4]


# 6. Feature coverage and value ranges
candidate_cols = [
    c for c in [
        "TMP_c", "DEW_c", "SLP_hpa",
        "VIS_m", "VIS_at_least_10mi",
        "CIG_m", "CIG_unlimited",
        "wind_direction_deg", "wind_speed_ms"
    ]
    if c in pilot_weather.columns
]

feature_summary = pd.DataFrame({
    "feature": candidate_cols,
    "missing_pct": [
        round(100 * pilot_weather[c].isna().mean(), 2)
        for c in candidate_cols
    ],
    "min": [pilot_weather[c].min() for c in candidate_cols],
    "median": [pilot_weather[c].median() for c in candidate_cols],
    "max": [pilot_weather[c].max() for c in candidate_cols],
})

display(feature_summary)


# 7. Quality-code counts
qc_columns = {
    "TMP": "TMP_qc",
    "DEW": "DEW_qc",
    "SLP": "SLP_qc",
    "VIS": "VIS_qc",
    "CIG": "CIG_qc",
    "WND direction": "wind_direction_qc",
    "WND speed": "wind_speed_qc",
}

for label, col in qc_columns.items():
    if col in pilot_weather.columns:
        print(f"\n{label} quality-code counts:")
        display(
            pilot_weather[col]
            .value_counts(dropna=False)
            .rename_axis("quality_code")
            .reset_index(name="count")
        )


,feature,missing_pct,min,median,max
0,TMP_c,0.10,10.0,27.2,46.7
1,DEW_c,0.10,-9.4,16.1,25.6
2,SLP_hpa,4.21,1000.7,1014.9,1028.1
3,VIS_m,0.08,200.0,16093.0,16093.0
4,VIS_at_least_10mi,0.00,0.0,1.0,1.0
5,CIG_m,74.10,30.0,3658.0,10668.0
6,CIG_unlimited,0.00,0.0,1.0,1.0
7,wind_direction_deg,14.85,10.0,200.0,360.0
8,wind_speed_ms,0.14,0.0,3.6,16.0



TMP quality-code counts:


,quality_code,count
0,5,148909
1,1,55686
2,A,217
3,<NA>,166
4,9,33
5,6,13



DEW quality-code counts:


,quality_code,count
0,5,148984
1,1,55656
2,<NA>,166
3,A,77
4,6,73
5,9,38
6,2,30


...

,quality_code,count
0,5,149473
1,1,28773
2,9,26414
3,6,198
4,<NA>,166



WND direction quality-code counts:


,quality_code,count
0,5,127554
1,1,47022
2,9,30280
3,<NA>,166
4,U,2



WND speed quality-code counts:


,quality_code,count
0,5,148918
1,1,55661
2,<NA>,166
3,A,150
4,9,127
5,U,2


In [34]:

# CELL 7 — Create quality-filtered weather candidates

weather_candidates = pilot_weather.copy()

# NOAA quality code "1" = passed all quality control checks.
# Keep only code "1" for these initial candidate measurements.
quality_rules = {
    "TMP": ("TMP_c", "TMP_qc"),
    "DEW": ("DEW_c", "DEW_qc"),
    "SLP": ("SLP_hpa", "SLP_qc"),
    "WND_speed": ("wind_speed_ms", "wind_speed_qc"),
}

for name, (value_col, qc_col) in quality_rules.items():
    if value_col in weather_candidates and qc_col in weather_candidates:
        weather_candidates[value_col + "_qc1"] = (
            weather_candidates[value_col].where(
                weather_candidates[qc_col].astype("string").eq("1")
            )
        )

# Wind direction only where its quality code is "1"
if {"wind_direction_deg", "wind_direction_qc"}.issubset(
    weather_candidates.columns
):
    weather_candidates["wind_direction_deg_qc1"] = (
        weather_candidates["wind_direction_deg"].where(
            weather_candidates["wind_direction_qc"]
            .astype("string").eq("1")
        )
    )

# Coverage after quality filtering
qc_features = [
    c for c in [
        "TMP_c_qc1", "DEW_c_qc1", "SLP_hpa_qc1",
        "wind_speed_ms_qc1", "wind_direction_deg_qc1"
    ]
    if c in weather_candidates.columns
]

display(pd.DataFrame({
    "feature": qc_features,
    "missing_pct_after_qc": [
        round(100 * weather_candidates[c].isna().mean(), 2)
        for c in qc_features
    ],
    "median": [
        weather_candidates[c].median()
        for c in qc_features
    ],
}))

print("Original pilot rows:", len(pilot_weather))
print("Candidate rows:", len(weather_candidates))
print("Original pilot weather data remains unchanged.")


,feature,missing_pct_after_qc,median
0,TMP_c_qc1,72.84,27.8
1,DEW_c_qc1,72.85,16.1
2,SLP_hpa_qc1,72.91,1014.9
3,wind_speed_ms_qc1,72.85,3.6
4,wind_direction_deg_qc1,77.07,200.0


Original pilot rows: 205024
Candidate rows: 205024
Original pilot weather data remains unchanged.


In [35]:

# CELL 8 — Compare strict QC vs provisional codes 1 + 5

qc_comparison = []

for name, (value_col, qc_col) in quality_rules.items():
    if value_col not in pilot_weather or qc_col not in pilot_weather:
        continue

    qc = pilot_weather[qc_col].astype("string")
    value = pilot_weather[value_col]

    qc_comparison.append({
        "feature": name,
        "original_missing_pct": round(100 * value.isna().mean(), 2),
        "code_1_coverage_pct": round(
            100 * (value.notna() & qc.eq("1")).mean(), 2
        ),
        "codes_1_and_5_coverage_pct": round(
            100 * (value.notna() & qc.isin(["1", "5"])).mean(), 2
        ),
        "qc_codes_seen": ", ".join(
            sorted(qc.dropna().unique().tolist())
        ),
    })

display(pd.DataFrame(qc_comparison))


,feature,original_missing_pct,code_1_coverage_pct,codes_1_and_5_coverage_pct,qc_codes_seen
0,TMP,0.10,27.16,99.79,"1, 5, 6, 9, A"
1,DEW,0.10,27.15,99.81,"1, 2, 5, 6, 9, A"
2,SLP,4.21,27.09,95.77,"1, 5, 6, 9"
3,WND_speed,0.14,27.15,99.78,"1, 5, 9, A, U"


In [36]:

# CELL 9 — Provisional weather features (keep QC codes 1 and 5)

weather_candidates = pilot_weather.copy()

quality_rules = {
    "TMP": ("TMP_c", "TMP_qc"),
    "DEW": ("DEW_c", "DEW_qc"),
    "SLP": ("SLP_hpa", "SLP_qc"),
    "WND_speed": ("wind_speed_ms", "wind_speed_qc"),
    "WND_direction": ("wind_direction_deg", "wind_direction_qc"),
}

for name, (value_col, qc_col) in quality_rules.items():
    if value_col not in weather_candidates or qc_col not in weather_candidates:
        continue

    qc = weather_candidates[qc_col].astype("string")
    valid_qc = qc.isin(["1", "5"])

    weather_candidates[f"{value_col}_candidate"] = (
        weather_candidates[value_col].where(valid_qc)
    )

# Exclude impossible/sentinel values from candidate features
if "SLP_hpa_candidate" in weather_candidates:
    weather_candidates["SLP_hpa_candidate"] = (
        weather_candidates["SLP_hpa_candidate"].where(
            weather_candidates["SLP_hpa_candidate"].between(800, 1100)
        )
    )

if "wind_direction_deg_candidate" in weather_candidates:
    weather_candidates["wind_direction_deg_candidate"] = (
        weather_candidates["wind_direction_deg_candidate"].where(
            weather_candidates["wind_direction_deg_candidate"].between(0, 360)
        )
    )

candidate_cols = [
    "TMP_c_candidate",
    "DEW_c_candidate",
    "SLP_hpa_candidate",
    "wind_speed_ms_candidate",
    "wind_direction_deg_candidate",
]

display(pd.DataFrame({
    "feature": candidate_cols,
    "missing_pct": [
        round(100 * weather_candidates[c].isna().mean(), 2)
        for c in candidate_cols
    ],
    "median": [
        weather_candidates[c].median()
        for c in candidate_cols
    ],
}))

print("Rows preserved:", len(weather_candidates))
print("Original pilot_weather remains unchanged.")


,feature,missing_pct,median
0,TMP_c_candidate,0.21,27.2
1,DEW_c_candidate,0.19,16.1
2,SLP_hpa_candidate,4.23,1014.9
3,wind_speed_ms_candidate,0.22,3.6
4,wind_direction_deg_candidate,14.85,200.0


Rows preserved: 205024
Original pilot_weather remains unchanged.


In [37]:

# CELL 10 — Sanity checks for weather candidates

check_cols = [
    "TMP_c_candidate",
    "DEW_c_candidate",
    "SLP_hpa_candidate",
    "wind_speed_ms_candidate",
    "wind_direction_deg_candidate",
    "VIS_m",
    "CIG_m",
]

range_rows = []

for col in check_cols:
    if col not in weather_candidates:
        continue

    s = weather_candidates[col].dropna()
    range_rows.append({
        "feature": col,
        "count": len(s),
        "min": s.min() if len(s) else None,
        "median": s.median() if len(s) else None,
        "max": s.max() if len(s) else None,
        "missing_pct": round(
            100 * weather_candidates[col].isna().mean(), 2
        ),
    })

display(pd.DataFrame(range_rows))

# Inspect raw encodings for visibility, ceiling, and wind
for col in ["VIS_raw", "CIG_raw", "WND_raw"]:
    if col in weather_candidates:
        print(f"\nMost common {col} values:")
        display(weather_candidates[col].value_counts(dropna=False).head(10))


,feature,count,min,median,max,missing_pct
0,TMP_c_candidate,204595,10.0,27.2,46.7,0.21
1,DEW_c_candidate,204640,-9.4,16.1,25.6,0.19
2,SLP_hpa_candidate,196342,1000.7,1014.9,1028.1,4.23
3,wind_speed_ms_candidate,204579,0.0,3.6,13.9,0.22
4,wind_direction_deg_candidate,174576,10.0,200.0,360.0,14.85
5,VIS_m,204858,200.0,16093.0,16093.0,0.08
6,CIG_m,53111,30.0,3658.0,10668.0,74.10



Most common VIS_raw values:


VIS_raw
016093,5,N,5    136363
016000,1,9,9     50886
014484,5,N,5      3030
012875,5,N,5      2342
011265,5,N,5      1860
014000,1,9,9      1386
009656,5,N,5      1130
008047,5,N,5      1080
016093,1,9,9       806
011000,1,9,9       631
Name: count, dtype: int64


Most common CIG_raw values:


CIG_raw
22000,5,9,N    96818
22000,1,9,N    28495
99999,9,9,N    26414
07620,5,M,N    10808
06706,5,M,N     2783
04572,5,M,N     2525
06096,5,M,N     2314
04267,5,M,N     2194
05486,5,M,N     1683
03353,5,M,N     1427
Name: count, dtype: int64


Most common WND_raw values:


WND_raw
999,9,C,0000,5    11556
999,9,C,0000,1     4762
999,9,V,0015,5     3786
999,9,V,0021,5     2634
999,9,V,0026,5     2192
999,9,N,0015,1     1746
999,9,V,0031,5     1346
260,5,N,0031,5     1084
999,9,N,0021,1     1005
320,5,N,0031,5      903
Name: count, dtype: int64

In [38]:

# CELL 11 — Handle visibility and ceiling special codes

# Parse raw visibility components
vis_parts = weather_candidates["VIS_raw"].astype("string").str.split(",", expand=True)
vis_value = pd.to_numeric(vis_parts[0], errors="coerce")
vis_qc = vis_parts[1].astype("string")

# Keep visibility codes 1 and 5; reject invalid/sentinel values
weather_candidates["visibility_m_candidate"] = vis_value.where(
    vis_qc.isin(["1", "5"]) & vis_value.between(0, 16093)
)

# Parse raw ceiling components
cig_parts = weather_candidates["CIG_raw"].astype("string").str.split(",", expand=True)
cig_value = pd.to_numeric(cig_parts[0], errors="coerce")
cig_qc = cig_parts[1].astype("string")

# Separate unlimited ceiling from actual ceiling height
weather_candidates["ceiling_unlimited"] = (
    cig_value.eq(22000) & cig_qc.isin(["1", "5"])
).astype("float")

weather_candidates.loc[
    ~cig_qc.isin(["1", "5"]), "ceiling_unlimited"
] = float("nan")

weather_candidates["ceiling_height_m_candidate"] = cig_value.where(
    cig_qc.isin(["1", "5"]) & cig_value.between(0, 20000)
)

# 22000 means unlimited, not a literal ceiling height
weather_candidates.loc[
    cig_value.eq(22000), "ceiling_height_m_candidate"
] = float("nan")

# 99999 is a missing-value code
weather_candidates.loc[
    cig_value.eq(99999), "ceiling_height_m_candidate"
] = float("nan")

display(pd.DataFrame({
    "feature": [
        "visibility_m_candidate",
        "ceiling_height_m_candidate",
        "ceiling_unlimited",
    ],
    "missing_pct": [
        round(100 * weather_candidates[c].isna().mean(), 2)
        for c in [
            "visibility_m_candidate",
            "ceiling_height_m_candidate",
            "ceiling_unlimited",
        ]
    ],
    "median": [
        weather_candidates[c].median()
        for c in [
            "visibility_m_candidate",
            "ceiling_height_m_candidate",
            "ceiling_unlimited",
        ]
    ],
}))

print("Rows preserved:", len(weather_candidates))


,feature,missing_pct,median
0,visibility_m_candidate,0.30,16093.0
1,ceiling_height_m_candidate,74.18,3658.0
2,ceiling_unlimited,13.06,1.0


Rows preserved: 205024


In [39]:

# CELL 12 — Final weather pilot validation

final_weather_features = [
    "TMP_c_candidate",
    "DEW_c_candidate",
    "SLP_hpa_candidate",
    "wind_speed_ms_candidate",
    "wind_direction_deg_candidate",
    "visibility_m_candidate",
    "ceiling_height_m_candidate",
    "ceiling_unlimited",
]

available_features = [
    c for c in final_weather_features if c in weather_candidates.columns
]

validation = pd.DataFrame({
    "feature": available_features,
    "missing_pct": [
        round(100 * weather_candidates[c].isna().mean(), 2)
        for c in available_features
    ],
    "unique_values": [
        weather_candidates[c].nunique(dropna=True)
        for c in available_features
    ],
})

display(validation)

print("Pilot rows:", len(weather_candidates))
print("Duplicate rows:", weather_candidates.duplicated().sum())
print("Missing flight IDs:", weather_candidates["flight_id"].isna().sum()
      if "flight_id" in weather_candidates.columns else "flight_id column not found")
print("Feature columns available:", len(available_features))


,feature,missing_pct,unique_values
0,TMP_c_candidate,0.21,88
1,DEW_c_candidate,0.19,83
2,SLP_hpa_candidate,4.23,256
3,wind_speed_ms_candidate,0.22,26
4,wind_direction_deg_candidate,14.85,36
5,visibility_m_candidate,0.30,35
6,ceiling_height_m_candidate,74.18,79
7,ceiling_unlimited,13.06,2


Pilot rows: 205024
Duplicate rows: 0
Missing flight IDs: flight_id column not found
Feature columns available: 8


In [40]:

# CELL 13 — Save the weather pilot artifact

from pathlib import Path

output_dir = Path("artifacts/weather_pilot")
output_dir.mkdir(parents=True, exist_ok=True)

# Keep original flight columns, weather join metadata, and candidate features.
weather_metadata = [
    c for c in [
        "weather_time_utc",
        "weather_age_minutes",
        "weather_cutoff_utc",
    ]
    if c in weather_candidates.columns
]

flight_columns = [
    c for c in pilot_flights.columns
    if c in weather_candidates.columns
]

save_columns = list(dict.fromkeys(
    flight_columns + weather_metadata + available_features
))

weather_pilot_final = weather_candidates[save_columns].copy()

output_path = output_dir / "flight_weather_pilot_aug2024.parquet"
weather_pilot_final.to_parquet(output_path, index=False)

print("Saved to:", output_path)
print("Rows:", len(weather_pilot_final))
print("Columns:", len(weather_pilot_final.columns))
print("Weather features:", available_features)
print("File size (MB):", round(output_path.stat().st_size / (1024**2), 2))


Saved to: artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet
Rows: 205024
Columns: 31
Weather features: ['TMP_c_candidate', 'DEW_c_candidate', 'SLP_hpa_candidate', 'wind_speed_ms_candidate', 'wind_direction_deg_candidate', 'visibility_m_candidate', 'ceiling_height_m_candidate', 'ceiling_unlimited']
File size (MB): 8.43


---

In [1]:
# SLOT 3 — Extend NOAA weather data through December 2024

import pandas as pd

station_ids = {
    "ATL": "72219013874",
    "DFW": "72259003927",
    "DEN": "72565003017",
    "ORD": "72530094846",
    "CLT": "72314013881",
    "LAX": "72295023174",
    "LAS": "72386023169",
    "SEA": "72793024233",
    "PHX": "72278023183",
    "LGA": "72503014732",
}

weather_parts = []

for airport, station_id in station_ids.items():
    url = (
        "https://noaa-global-hourly-pds.s3.amazonaws.com/"
        f"2024/{station_id}.csv"
    )

    df = pd.read_csv(url, low_memory=False)
    df["DATE"] = pd.to_datetime(df["DATE"], utc=True, errors="coerce")

    df = df[
        df["DATE"].ge("2024-09-01",) &
        df["DATE"].lt("2025-01-01")
    ].copy()

    df["station_id"] = station_id
    df["airport"] = airport
    weather_parts.append(df)

weather_sep_dec_raw = pd.concat(weather_parts, ignore_index=True)

print("Rows:", len(weather_sep_dec_raw))
print("Stations:", weather_sep_dec_raw["station_id"].nunique())
print("\nRows by month:")
display(
    weather_sep_dec_raw.groupby(
        weather_sep_dec_raw["DATE"].dt.to_period("M")
    ).size().rename("observations").to_frame()
)


Rows: 44102
Stations: 10

Rows by month:


/var/folders/h6/8v7ghg7d5pzgkx9ldll6y32m0000gn/T/ipykernel_13940/2061867028.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  weather_sep_dec_raw["DATE"].dt.to_period("M")


,observations
DATE,
2024-09,10883
2024-10,10979
2024-11,10929
2024-12,11311


In [3]:
# SLOT 3 — Find existing flight datasets and split variables

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)
        if {"FlightDate", "Origin"}.issubset(cols):
            print(
                f"{name}: rows={len(obj):,}, "
                f"date_min={obj['FlightDate'].min()}, "
                f"date_max={obj['FlightDate'].max()}, "
                f"origins={obj['Origin'].nunique()}"
            )

print("\nExisting split variables:")
for name in list(globals()):
    if any(word in name.lower() for word in ["train", "valid", "test"]):
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(name, len(obj))



Existing split variables:


In [5]:
# SLOT 3 — Find flight-related DataFrames

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")
        print("  Columns:", list(obj.columns[:15]))


df: (4301, 103)
  Columns: ['STATION', 'DATE', 'SOURCE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'CALL_SIGN', 'QUALITY_CONTROL', 'WND', 'CIG', 'VIS', 'TMP', 'DEW']
weather_sep_dec_raw: (44102, 108)
  Columns: ['STATION', 'DATE', 'SOURCE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'CALL_SIGN', 'QUALITY_CONTROL', 'WND', 'CIG', 'VIS', 'TMP', 'DEW']


In [6]:

from pathlib import Path

for pattern in ["*.parquet", "**/*.parquet"]:
    print(f"\n{pattern}")
    for p in Path(".").glob(pattern):
        print(p)



*.parquet

**/*.parquet
artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet


In [7]:

from pathlib import Path

root = Path.cwd()
print("Notebook directory:", root)

for folder in ["data", "datasets", "artifacts", "processed"]:
    p = root / folder
    if p.exists():
        print(f"\n{p}/")
        for f in p.rglob("*"):
            if f.is_file() and f.suffix.lower() in {".parquet", ".csv", ".feather", ".pkl"}:
                print(f, f.stat().st_size // 1024, "KB")


Notebook directory: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/src/notebooks

/Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/src/notebooks/artifacts/
/Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/src/notebooks/artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet 8632 KB


In [8]:

from pathlib import Path
import pandas as pd

# Current notebook is inside src/notebooks
PROJECT_ROOT = Path.cwd().parents[1]

print("PROJECT ROOT:", PROJECT_ROOT)

ignored_dirs = {
    ".git", ".venv", "venv", "__pycache__",
    ".ipynb_checkpoints", "node_modules"
}

records = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(PROJECT_ROOT)

    if any(part in ignored_dirs for part in relative.parts):
        continue

    records.append({
        "type": path.suffix.lower() or "[no extension]",
        "path": str(relative),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2),
        "modified": pd.Timestamp(path.stat().st_mtime, unit="s")
            .strftime("%Y-%m-%d %H:%M"),
    })

inventory = pd.DataFrame(records)

if inventory.empty:
    print("No files found.")
else:
    inventory = inventory.sort_values(
        ["type", "path"]
    ).reset_index(drop=True)

    display(inventory)
    print("\nTotal files:", len(inventory))
    print("Total size (MB):", round(inventory["size_mb"].sum(), 2))

    print("\nFILES BY TYPE:")
    display(
        inventory.groupby("type")
        .agg(files=("path", "count"), total_mb=("size_mb", "sum"))
        .sort_values("total_mb", ascending=False)
    )


PROJECT ROOT: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred


,type,path,size_mb,modified
0,.csv,src/data/external/noaa_isd/72219013874_2024.csv,0.04,2026-10-22 10:25
1,.csv,src/data/external/noaa_isd/72259003927_2024.csv,0.04,2026-10-22 10:25
2,.csv,src/data/external/noaa_isd/72278023183_2024.csv,0.04,2026-10-22 10:25
3,.csv,src/data/external/noaa_isd/72295023174_2024.csv,0.04,2026-10-22 10:25
4,.csv,src/data/external/noaa_isd/72314013881_2024.csv,0.04,2026-10-22 10:25
...,...,...,...,...
58,.zip,src/data/raw/2024/On_Time_Reporting_Carrier_On...,29.43,2026-10-01 19:38
59,.zip,src/data/raw/2024/On_Time_Reporting_Carrier_On...,27.12,2026-10-01 19:39
60,[no extension],.DS_Store,0.01,2026-09-21 00:16
61,[no extension],.gitignore,0.00,2026-09-22 14:37



Total files: 63
Total size (MB): 3722.46

FILES BY TYPE:


,files,total_mb
type,,
.csv,24,3292.16
.zip,12,337.92
.parquet,2,91.39
.ipynb,4,0.60
.lock,1,0.24
.html,12,0.12
.md,2,0.02
[no extension],3,0.01
.toml,1,0.00


In [9]:

from pathlib import Path
import json
import re

root = Path.cwd().parents[1]
ignored = {".git", ".venv", "venv", "__pycache__", ".ipynb_checkpoints"}

files = sorted(
    p for p in root.rglob("*")
    if p.is_file() and not any(part in ignored for part in p.relative_to(root).parts)
)

print("=== COMPLETE PROJECT INVENTORY ===")
for p in files:
    print(f"{p.stat().st_size / (1024**2):8.2f} MB  {p.relative_to(root)}")

print("\n=== NOTEBOOK / SCRIPT DATA REFERENCES ===")
for p in files:
    if p.suffix == ".ipynb":
        try:
            notebook = json.loads(p.read_text(encoding="utf-8"))
            source = "\n".join(
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
            )
        except Exception as e:
            print(f"\n{p.relative_to(root)}: could not read: {e}")
            continue
    elif p.suffix == ".py":
        source = p.read_text(encoding="utf-8", errors="replace")
    else:
        continue

    matches = [
        (i, line.strip()) for i, line in enumerate(source.splitlines(), 1)
        if re.search(
            r"\.csv|\.parquet|\.zip|read_csv|read_parquet|to_parquet|"
            r"data/|artifacts/|Path\(|DATA_DIR|DATA_ROOT",
            line,
            re.IGNORECASE,
        )
    ]

    print(f"\n--- {p.relative_to(root)} ---")
    for i, line in matches:
        print(f"{i}: {line[:220]}")


=== COMPLETE PROJECT INVENTORY ===
    0.01 MB  .DS_Store
    0.00 MB  .gitignore
    0.00 MB  .python-version
    0.00 MB  README.md
    0.02 MB  docs/FlightDelay_Intelligence_Project_Specification.md
    0.00 MB  main.py
    0.00 MB  pyproject.toml
    0.00 MB  src/app.py
    0.04 MB  src/data/external/noaa_isd/72219013874_2024.csv
    0.04 MB  src/data/external/noaa_isd/72259003927_2024.csv
    0.04 MB  src/data/external/noaa_isd/72278023183_2024.csv
    0.04 MB  src/data/external/noaa_isd/72295023174_2024.csv
    0.04 MB  src/data/external/noaa_isd/72314013881_2024.csv
    0.04 MB  src/data/external/noaa_isd/72386023169_2024.csv
    0.04 MB  src/data/external/noaa_isd/72503014732_2024.csv
    0.04 MB  src/data/external/noaa_isd/72530094846_2024.csv
    0.04 MB  src/data/external/noaa_isd/72565003017_2024.csv
    0.04 MB  src/data/external/noaa_isd/72793024233_2024.csv
    2.78 MB  src/data/external/noaa_isd/isd-history.csv
   82.96 MB  src/data/processed/flights_2024.parquet
   26.

In [10]:

from pathlib import Path

root = Path.cwd().parents[1]

for folder in ["src/data", "src/notebooks", "src", "artifacts"]:
    base = root / folder
    if not base.exists():
        continue

    print(f"\n## {folder}/")
    for p in sorted(base.rglob("*")):
        if p.is_file() and not any(
            part in {".ipynb_checkpoints", "__pycache__"}
            for part in p.parts
        ):
            print(f"{p.stat().st_size / 1024**2:7.2f} MB  {p.relative_to(root)}")



## src/data/
   0.04 MB  src/data/external/noaa_isd/72219013874_2024.csv
   0.04 MB  src/data/external/noaa_isd/72259003927_2024.csv
   0.04 MB  src/data/external/noaa_isd/72278023183_2024.csv
   0.04 MB  src/data/external/noaa_isd/72295023174_2024.csv
   0.04 MB  src/data/external/noaa_isd/72314013881_2024.csv
   0.04 MB  src/data/external/noaa_isd/72386023169_2024.csv
   0.04 MB  src/data/external/noaa_isd/72503014732_2024.csv
   0.04 MB  src/data/external/noaa_isd/72530094846_2024.csv
   0.04 MB  src/data/external/noaa_isd/72565003017_2024.csv
   0.04 MB  src/data/external/noaa_isd/72793024233_2024.csv
   2.78 MB  src/data/external/noaa_isd/isd-history.csv
  82.96 MB  src/data/processed/flights_2024.parquet
  26.30 MB  src/data/raw/2024/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
  28.06 MB  src/data/raw/2024/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip
  26.85 MB  src/data/raw/2024/On_Time_Reporting_Carrier_On_Time_Performance_19

In [11]:

from pathlib import Path
import shutil

root = Path.cwd().parents[1]

source = root / "src/notebooks/artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet"
destination_dir = root / "artifacts/weather_pilot"
destination = destination_dir / source.name

if not source.exists():
    print("Source not found:", source)
elif destination.exists():
    print("Destination already exists; nothing moved:", destination)
else:
    destination_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(source), str(destination))
    print("Moved successfully:")
    print("From:", source.relative_to(root))
    print("To:  ", destination.relative_to(root))

    # Remove only the empty directories left behind
    old_dir = source.parent
    if old_dir.exists() and not any(old_dir.iterdir()):
        old_dir.rmdir()
        parent = old_dir.parent
        if parent.name == "artifacts" and parent.exists() and not any(parent.iterdir()):
            parent.rmdir()

    print("Destination size (MB):", round(destination.stat().st_size / 1024**2, 2))


Moved successfully:
From: src/notebooks/artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet
To:   artifacts/weather_pilot/flight_weather_pilot_aug2024.parquet
Destination size (MB): 8.43


In [15]:

from pathlib import Path
import pandas as pd

root = Path.cwd().parents[1]

rows = []
for p in sorted(root.rglob("*")):
    if not p.is_file():
        continue
    if any(part in {".git", ".ipynb_checkpoints", "__pycache__", ".venv"}
           for part in p.parts):
        continue

    rel = p.relative_to(root)
    suffix = p.suffix.lower()

    if suffix in {".zip"}:
        category = "raw_download"
    elif suffix in {".parquet", ".feather"}:
        category = "processed_or_generated_data"
    elif suffix in {".ipynb"}:
        category = "notebook"
    elif suffix in {".html", ".htm"}:
        category = "html_export_or_report"
    elif suffix in {".csv", ".tsv"}:
        category = "tabular_data"
    elif suffix in {".py"}:
        category = "python_code"
    elif suffix in {".md", ".rst", ".txt", ".pdf"}:
        category = "documentation_or_report"
    else:
        category = "other"

    rows.append({
        "path": str(rel),
        "extension": suffix or "[no extension]",
        "category": category,
        "size_mb": round(p.stat().st_size / 1024**2, 3),
    })

inventory = pd.DataFrame(rows)
report = root / "project_file_inventory.csv"
inventory.to_csv(report, index=False)

print(f"Project: {root}")
print(f"Files inventoried: {len(inventory)}")
print(f"Inventory saved to: {report}")
print("\nFile counts:")
print(inventory.groupby(["category", "extension"]).size().to_string())
print("\nLargest files:")
print(
    inventory.sort_values("size_mb", ascending=False)
    [["size_mb", "path"]]
    .head(15)
    .to_string(index=False)
)


Project: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred
Files inventoried: 63
Inventory saved to: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/project_file_inventory.csv

File counts:
category                     extension     
documentation_or_report      .md                2
html_export_or_report        .html             12
notebook                     .ipynb             4
other                        .lock              1
                             .toml              1
                             [no extension]     3
processed_or_generated_data  .parquet           2
python_code                  .py                2
raw_download                 .zip              12
tabular_data                 .csv              24

Largest files:
 size_mb                                                                                                path
 274.168   src/data/raw/2024/month_7/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)

In [16]:

from pathlib import Path
import json
import re

root = Path.cwd().parents[1]

print("=== HTML FILES ===")
for p in sorted(root.rglob("*.html")):
    if not any(x in {".git", ".venv", "__pycache__"} for x in p.parts):
        print(p.relative_to(root))

print("\n=== CSV FILES BY FOLDER ===")
csvs = [
    p for p in root.rglob("*.csv")
    if not any(x in {".git", ".venv", "__pycache__"} for x in p.parts)
]
by_folder = {}
for p in csvs:
    by_folder.setdefault(str(p.parent.relative_to(root)), []).append(p)

for folder, files in sorted(by_folder.items()):
    total_mb = sum(p.stat().st_size for p in files) / 1024**2
    print(f"\n{folder}/ — {len(files)} files, {total_mb:.1f} MB")
    for p in sorted(files):
        print(f"  {p.name}")

print("\n=== REFERENCES TO DATA / ARTIFACT PATHS ===")
pattern = re.compile(
    r"src/data|src/notebooks/artifacts|artifacts/|\\.csv|\\.parquet|\\.html|\\.zip",
    re.IGNORECASE
)

for p in sorted(root.rglob("*")):
    if not p.is_file() or p.suffix not in {".py", ".ipynb", ".md", ".toml"}:
        continue
    if any(x in {".git", ".venv", "__pycache__"} for x in p.parts):
        continue

    try:
        if p.suffix == ".ipynb":
            nb = json.loads(p.read_text(encoding="utf-8"))
            lines = []
            for i, cell in enumerate(nb.get("cells", [])):
                if cell.get("cell_type") == "code":
                    source = "".join(cell.get("source", []))
                    for line_no, line in enumerate(source.splitlines(), 1):
                        if pattern.search(line):
                            lines.append(f"  cell {i}, line {line_no}: {line.strip()}")
        else:
            lines = [
                f"  line {i}: {line.strip()}"
                for i, line in enumerate(p.read_text(
                    encoding="utf-8", errors="ignore"
                ).splitlines(), 1)
                if pattern.search(line)
            ]

        if lines:
            print(f"\n{p.relative_to(root)}")
            for line in lines[:30]:
                print(line)
            if len(lines) > 30:
                print(f"  ... {len(lines) - 30} more matches")
    except Exception as e:
        print(f"Could not inspect {p.relative_to(root)}: {e}")


=== HTML FILES ===
src/data/raw/2024/month_1/readme.html
src/data/raw/2024/month_10/readme.html
src/data/raw/2024/month_11/readme.html
src/data/raw/2024/month_12/readme.html
src/data/raw/2024/month_2/readme.html
src/data/raw/2024/month_3/readme.html
src/data/raw/2024/month_4/readme.html
src/data/raw/2024/month_5/readme.html
src/data/raw/2024/month_6/readme.html
src/data/raw/2024/month_7/readme.html
src/data/raw/2024/month_8/readme.html
src/data/raw/2024/month_9/readme.html

=== CSV FILES BY FOLDER ===

./ — 1 files, 0.0 MB
  project_file_inventory.csv

src/data/external/noaa_isd/ — 11 files, 3.1 MB
  72219013874_2024.csv
  72259003927_2024.csv
  72278023183_2024.csv
  72295023174_2024.csv
  72314013881_2024.csv
  72386023169_2024.csv
  72503014732_2024.csv
  72530094846_2024.csv
  72565003017_2024.csv
  72793024233_2024.csv
  isd-history.csv

src/data/raw/ — 1 files, 235.4 MB
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv

src/data/raw/2024/month_1/ — 1 files

In [17]:
from pathlib import Path
import json
import re

root = Path.cwd().parents[1]

pattern = re.compile(
    r"src/data|src/notebooks/artifacts|artifacts/|"
    r"\.csv|\.parquet|\.html|\.zip",
    re.IGNORECASE,
)

print("=== NOTEBOOKS AND SCRIPTS WITH PATH REFERENCES ===")

for p in sorted(root.rglob("*")):
    if not p.is_file() or p.suffix not in {".py", ".ipynb", ".toml"}:
        continue
    if any(part in {".git", ".venv", "__pycache__"} for part in p.parts):
        continue

    if p.suffix == ".ipynb":
        try:
            nb = json.loads(p.read_text(encoding="utf-8"))
            sources = [
                (i, "".join(cell.get("source", [])))
                for i, cell in enumerate(nb.get("cells", []))
                if cell.get("cell_type") == "code"
            ]
        except Exception:
            continue
    else:
        sources = [(None, p.read_text(encoding="utf-8", errors="ignore"))]

    matches = []
    for cell_no, source in sources:
        for line_no, line in enumerate(source.splitlines(), 1):
            if pattern.search(line):
                location = f"cell {cell_no}" if cell_no is not None else f"line {line_no}"
                matches.append(f"{location}: {line.strip()[:160]}")

    if matches:
        print(f"\n{p.relative_to(root)} ({len(matches)} matches)")
        for line in matches[:5]:
            print(" ", line)
        if len(matches) > 5:
            print(f"  ... plus {len(matches) - 5} matches")

=== NOTEBOOKS AND SCRIPTS WITH PATH REFERENCES ===

src/notebooks/flight_delay_predV1.0.ipynb (1 matches)
  cell 4: DATASET_PATH = "../data/raw/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv"

src/notebooks/flight_delay_predV1.1.ipynb (3 matches)
  cell 1: zip_files = sorted(raw_dir.glob("*.zip"))
  cell 3: with zipfile.ZipFile(zip_path, "r") as z:
  cell 5: csv_files = sorted(raw_dir.glob("month_*/*.csv"))

src/notebooks/flight_delay_predV1.2.ipynb (3 matches)
  cell 1: output_path = processed_dir / "flights_2024.parquet"
  cell 1: csv_files = sorted(raw_dir.glob("month_*/*.csv"))
  cell 3: parquet_path = Path("../data/processed/flights_2024.parquet")

src/notebooks/flight_delay_predV1.3.ipynb (22 matches)
  cell 1: data_path = Path("../data/processed/flights_2024.parquet")
  cell 25: "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
  cell 25: inventory_path = weather_dir / "isd-history.csv"
  cell 26: url = f"https://noaa-global-hourly-pds.s3.amazona

In [18]:
from pathlib import Path

root = Path.cwd().parents[1]

for folder in [root, root / "src", root / "src/notebooks", root / "artifacts"]:
    if not folder.exists():
        continue

    print(f"\n=== {folder.relative_to(root) or '.'} ===")

    for p in sorted(folder.iterdir(), key=lambda x: (x.is_file(), x.name.lower())):
        if p.is_dir():
            count = sum(1 for x in p.rglob("*") if x.is_file())
            print(f"[DIR]  {p.name}/ ({count} files)")
        else:
            size_mb = p.stat().st_size / (1024 ** 2)
            print(f"[FILE] {p.name} ({size_mb:.2f} MB)")


=== . ===
[DIR]  .git/ (117 files)
[DIR]  .venv/ (15872 files)
[DIR]  artifacts/ (1 files)
[DIR]  docs/ (1 files)
[DIR]  src/ (54 files)
[FILE] .DS_Store (0.01 MB)
[FILE] .gitignore (0.00 MB)
[FILE] .python-version (0.00 MB)
[FILE] main.py (0.00 MB)
[FILE] project_file_inventory.csv (0.01 MB)
[FILE] pyproject.toml (0.00 MB)
[FILE] README.md (0.00 MB)
[FILE] uv.lock (0.24 MB)

=== src ===
[DIR]  data/ (49 files)
[DIR]  notebooks/ (4 files)
[FILE] app.py (0.00 MB)

=== src/notebooks ===
[FILE] flight_delay_predV1.0.ipynb (0.32 MB)
[FILE] flight_delay_predV1.1.ipynb (0.04 MB)
[FILE] flight_delay_predV1.2.ipynb (0.05 MB)
[FILE] flight_delay_predV1.3.ipynb (0.24 MB)

=== artifacts ===
[DIR]  weather_pilot/ (1 files)


In [19]:
from pathlib import Path
import shutil

root = Path.cwd().parents[1]

source = root / "src/notebooks/project_file_inventory.csv"
destination_dir = root / "reports"
destination = destination_dir / "project_file_inventory.csv"

destination_dir.mkdir(exist_ok=True)

if source.exists():
    if destination.exists():
        print(f"STOP: destination already exists: {destination}")
    else:
        shutil.move(str(source), str(destination))
        print(f"Moved inventory report to: {destination.relative_to(root)}")
else:
    print("Inventory report is not in src/notebooks; no files changed.")

Inventory report is not in src/notebooks; no files changed.


In [20]:
from pathlib import Path

root = Path.cwd().parents[1]

for name in ["project_file_inventory.csv"]:
    matches = list(root.rglob(name))
    if matches:
        for p in matches:
            print(p.relative_to(root))
    else:
        print(f"{name}: not found")

project_file_inventory.csv
